In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

SCHEMA = "workspace.metropulse"
events = spark.table(f"{SCHEMA}.silver_stop_events")
routes = spark.table(f"{SCHEMA}.silver_routes")

def save(df, name):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{SCHEMA}.{name}")
    return spark.table(f"{SCHEMA}.{name}")

trips = events.groupBy("trip_id", "route_id", "direction_id").agg(
    F.min("departure_secs").alias("start_secs"),
    F.max("arrival_secs").alias("end_secs"),
)

In [0]:
def hhmm(col):
    return F.format_string("%02d:%02d",
                           (F.floor(col / 3600) % 24).cast("int"),
                           (F.floor((col % 3600) / 60)).cast("int"))

route_stats = trips.groupBy("route_id").agg(
    F.count("*").alias("weekday_trips"),
    F.round(F.avg((F.col("end_secs") - F.col("start_secs")) / 60), 1).alias("avg_trip_minutes"),
    F.min("start_secs").alias("first_secs"),
    F.max("start_secs").alias("last_secs"),
)

stops_per_route = events.groupBy("route_id").agg(F.countDistinct("stop_id").alias("stops_served"))

In [0]:
MIDDAY_START, MIDDAY_END = 10 * 3600, 14 * 3600

w = Window.partitionBy("route_id", "direction_id").orderBy("start_secs")

gaps = (
    trips.filter((F.col("start_secs") >= MIDDAY_START) & (F.col("start_secs") < MIDDAY_END))
    .withColumn("gap_min", (F.col("start_secs") - F.lag("start_secs").over(w)) / 60)
)

headway = (
    gaps.groupBy("route_id", "direction_id")
    .agg(F.percentile_approx("gap_min", 0.5).alias("median_gap"))
    .groupBy("route_id")
    .agg(F.max("median_gap").alias("midday_headway_min"))
)

In [0]:
gold_route_summary = save(
    routes
    .join(route_stats, "route_id", "inner")
    .join(stops_per_route, "route_id", "left")
    .join(headway, "route_id", "left")
    .withColumn("first_departure", hhmm(F.col("first_secs")))
    .withColumn("last_departure", hhmm(F.col("last_secs")))
    .withColumn("service_span_hours", F.round((F.col("last_secs") - F.col("first_secs")) / 3600, 1))
    .withColumn("frequent_midday", F.coalesce(F.col("midday_headway_min") <= 15, F.lit(False)))
    .drop("first_secs", "last_secs"),
    "gold_route_summary",
)
display(gold_route_summary.orderBy(F.desc("weekday_trips")))

route_id,route_short_name,route_long_name,route_type,mode,weekday_trips,avg_trip_minutes,stops_served,midday_headway_min,first_departure,last_departure,service_span_hours,frequent_midday
700,700,METRORAIL RED LINE,0,Rail,348,52.0,50,6.0,03:29,00:24,20.9,true
082,082,Westheimer,3,Bus,293,83.9,195,8.0,03:50,01:35,21.8,true
900,900,METRORAIL PURPLE LINE,0,Rail,214,26.5,26,12.0,03:25,01:30,22.1,true
800,800,METRORAIL GREEN LINE,0,Rail,205,21.2,18,12.0,03:30,00:45,21.3,true
056,056,Airline / Montrose,3,Bus,202,68.7,209,10.0,04:20,00:04,19.7,true
073,073,Bellfort,3,Bus,188,31.9,99,12.0,04:24,00:20,19.9,true
002,002,Bellaire,3,Bus,183,63.1,150,15.0,03:50,01:00,21.2,true
004,004,Beechnut,3,Bus,172,94.7,233,15.0,04:30,00:45,20.3,true
054,054,Scott,3,Bus,172,60.7,148,15.0,03:55,01:00,21.1,true
025,025,Richmond,3,Bus,168,84.2,234,15.0,04:05,00:43,20.6,true


In [0]:
gold_stop_summary = save(
    events.groupBy("stop_id", "stop_name", "stop_lat", "stop_lon").agg(
        F.countDistinct("route_id").alias("routes_served"),
        F.count("*").alias("weekday_departures"),
    ),
    "gold_stop_summary",
)

gold_hourly_service = save(
    trips.withColumn("start_hour", (F.floor(F.col("start_secs") / 3600) % 24).cast("int"))
    .join(routes, "route_id")
    .groupBy("route_id", "route_short_name", "route_long_name", "mode", "start_hour")
    .agg(F.count("*").alias("trips_started")),
    "gold_hourly_service",
)

In [0]:
total_route_trips = gold_route_summary.agg(F.sum("weekday_trips")).first()[0]
total_hourly_trips = gold_hourly_service.agg(F.sum("trips_started")).first()[0]

print(f"routes running:      {gold_route_summary.count():,}")
print(f"trips (route table): {total_route_trips:,}")
print(f"trips (hourly table):{total_hourly_trips:,}")
print(f"stops served:        {gold_stop_summary.count():,}")
print(f"frequent routes:     {gold_route_summary.filter('frequent_midday').count():,}")

assert total_route_trips == 9160 and total_hourly_trips == 9160, "Gold trip totals don't match Silver"
print("\nGold checks passed")

routes running:      109
trips (route table): 9,160
trips (hourly table):9,160
stops served:        8,791
frequent routes:     22

Gold checks passed
